In [ ]:
#| default_exp data

# Data

> S3, DynamoDB, RDS PostgreSQL, and ElastiCache Redis for GenAI data infrastructure.

In [ ]:
#| export
import secrets as _secrets_mod

## Amazon S3

Equivalent to Azure Blob Storage. Public access blocked, SSE-S3 encryption, versioning on by default.

```python
create_bucket(auth, 'my-docs-bucket', **HIPAA)
print(presigned_url(auth, 'my-docs-bucket', 'file.pdf', hours=2))
```

In [ ]:
#| export
_tags = lambda d: [{'Key': k, 'Value': v} for k, v in (d or {}).items()]

def _s3(auth):
    return auth.session.client('s3')

def create_bucket(auth, name, versioning=True, ssl_only=False,
                  access_logging=None, tags=None, **compliance_opts) -> dict:
    'Create S3 bucket. Blocks public access, enables SSE-S3 encryption. ssl_only=True adds a deny-HTTP bucket policy.'
    s3 = _s3(auth)
    kwargs = {'Bucket': name}
    if auth.region != 'us-east-1':
        kwargs['CreateBucketConfiguration'] = {'LocationConstraint': auth.region}
    try:
        s3.create_bucket(**kwargs)
    except s3.exceptions.BucketAlreadyOwnedByYou:
        pass
    s3.put_public_access_block(
        Bucket=name,
        PublicAccessBlockConfiguration={
            'BlockPublicAcls': True, 'IgnorePublicAcls': True,
            'BlockPublicPolicy': True, 'RestrictPublicBuckets': True,
        })
    s3.put_bucket_encryption(
        Bucket=name,
        ServerSideEncryptionConfiguration={
            'Rules': [{'ApplyServerSideEncryptionByDefault':
                       {'SSEAlgorithm': 'AES256'}}]})
    if versioning:
        s3.put_bucket_versioning(
            Bucket=name, VersioningConfiguration={'Status': 'Enabled'})
    if ssl_only or compliance_opts.get('ssl_only'):
        import json as _json
        s3.put_bucket_policy(Bucket=name, Policy=_json.dumps({
            'Version': '2012-10-17',
            'Statement': [{
                'Sid': 'DenyNonTLS',
                'Effect': 'Deny',
                'Principal': '*',
                'Action': 's3:*',
                'Resource': [f'arn:aws:s3:::{name}', f'arn:aws:s3:::{name}/*'],
                'Condition': {'Bool': {'aws:SecureTransport': 'false'}},
            }],
        }))
    if access_logging or compliance_opts.get('access_logging'):
        log_bucket = access_logging if isinstance(access_logging, str) else f'{name}-logs'
        s3.put_bucket_logging(
            Bucket=name,
            BucketLoggingStatus={
                'LoggingEnabled': {'TargetBucket': log_bucket,
                                   'TargetPrefix': f'{name}/'}})
    if tags:
        s3.put_bucket_tagging(
            Bucket=name,
            Tagging={'TagSet': _tags(tags)})
    return {'BucketName': name, 'Region': auth.region}

def bucket_url(name, key) -> str:
    'Return the s3:// URI for a bucket/key.'
    return f's3://{name}/{key}'

def presigned_url(auth, name, key, hours=1) -> str:
    'Generate a presigned GET URL for a S3 object.'
    return _s3(auth).generate_presigned_url(
        'get_object', Params={'Bucket': name, 'Key': key},
        ExpiresIn=hours * 3600)

def bucket_conn(name) -> str:
    'Return the bucket name (use with boto3 resource/client directly).'
    return name


## Amazon DynamoDB

Equivalent to Cosmos DB (NoSQL). PAY_PER_REQUEST billing, PITR on by default.

```python
create_table(auth, 'my-table', partition_key='id', **ISO27001)
```

In [ ]:
#| export
def _dynamo(auth):
    return auth.session.client('dynamodb')

def create_table(auth, name, partition_key, sort_key=None,
                 billing_mode='PAY_PER_REQUEST', tags=None, **compliance_opts) -> dict:
    'Create DynamoDB table with point-in-time recovery enabled.'
    client = _dynamo(auth)
    key_schema = [{'AttributeName': partition_key, 'KeyType': 'HASH'}]
    attr_defs = [{'AttributeName': partition_key, 'AttributeType': 'S'}]
    if sort_key:
        key_schema.append({'AttributeName': sort_key, 'KeyType': 'RANGE'})
        attr_defs.append({'AttributeName': sort_key, 'AttributeType': 'S'})
    try:
        resp = client.create_table(
            TableName=name,
            KeySchema=key_schema,
            AttributeDefinitions=attr_defs,
            BillingMode=billing_mode,
            Tags=_tags(tags),
        )['TableDescription']
    except client.exceptions.ResourceInUseException:
        resp = client.describe_table(TableName=name)['Table']
    client.update_continuous_backups(
        TableName=name,
        PointInTimeRecoverySpecification={'PointInTimeRecoveryEnabled': True})
    return resp

def table_resource(auth, name):
    'Return a boto3 DynamoDB Table resource for direct operations.'
    return auth.session.resource('dynamodb').Table(name)

def dynamo_conn(auth, name) -> str:
    'Return the table name (use with boto3 DynamoDB client/resource directly).'
    return name


## Amazon RDS PostgreSQL

Equivalent to Azure PostgreSQL Flexible Server. Encryption-at-rest enabled, deletion protection on for HIPAA/SOC2.

```python
create_postgres(auth, 'my-pg', **HIPAA)
print(postgres_conn(auth, 'my-pg'))
```

In [ ]:
#| export
def _rds(auth):
    return auth.session.client('rds')

def create_postgres(auth, name, instance_class='db.t3.medium', engine_version='16.3',
                    master_username='pgadmin', master_password=None,
                    multi_az=False, deletion_protection=False,
                    tags=None, **compliance_opts) -> dict:
    'Create RDS PostgreSQL. Encryption-at-rest, PubliclyAccessible=False. Auto-stores generated password in Secrets Manager at rds/{name}/master on first creation.'
    import json as _json
    from .network import create_secret
    client = _rds(auth)
    password = master_password or _secrets_mod.token_urlsafe(24)
    created_new = False
    try:
        resp = client.create_db_instance(
            DBInstanceIdentifier=name,
            DBInstanceClass=instance_class,
            Engine='postgres',
            EngineVersion=engine_version,
            MasterUsername=master_username,
            MasterUserPassword=password,
            MultiAZ=multi_az,
            StorageEncrypted=True,
            PubliclyAccessible=False,
            DeletionProtection=deletion_protection or compliance_opts.get('deletion_protection', False),
            AllocatedStorage=20,
            BackupRetentionPeriod=compliance_opts.get('backup_retention', 7),
            CACertificateIdentifier='rds-ca-rsa2048-g1',
            Tags=_tags(tags),
        )['DBInstance']
        created_new = True
    except client.exceptions.DBInstanceAlreadyExistsFault:
        resp = client.describe_db_instances(
            DBInstanceIdentifier=name)['DBInstances'][0]
    # Store generated password in Secrets Manager on first creation
    if created_new and not master_password:
        create_secret(auth, f'rds/{name}/master',
                      _json.dumps({'username': master_username, 'password': password}))
    return resp

def postgres_conn(auth, name, db='postgres') -> str:
    'Return a postgresql:// connection string (password not included — use Secrets Manager at rds/{name}/master).'
    inst = _rds(auth).describe_db_instances(
        DBInstanceIdentifier=name)['DBInstances'][0]
    host = inst['Endpoint']['Address']
    port = inst['Endpoint']['Port']
    user = inst['MasterUsername']
    return f'postgresql://{user}@{host}:{port}/{db}'


## Amazon ElastiCache Redis

Equivalent to Azure Cache for Redis. In-transit encryption enforced. Used for semantic LLM response caching.

```python
create_redis(auth, 'my-redis', **ISO27001)
print(redis_conn(auth, 'my-redis'))
```

In [ ]:
#| export
def _elasticache(auth):
    return auth.session.client('elasticache')

def create_redis(auth, name, node_type='cache.t3.micro', num_shards=1,
                 auth_token=False, tags=None, **compliance_opts) -> dict:
    'Create ElastiCache Redis OSS cluster. TLS + at-rest encryption. auth_token=True generates and stores AUTH token in Secrets Manager.'
    from .network import create_secret
    client = _elasticache(auth)
    kwargs = dict(
        ReplicationGroupId=name,
        ReplicationGroupDescription=name,
        CacheNodeType=node_type,
        Engine='redis',
        NumNodeGroups=num_shards,
        ReplicasPerNodeGroup=0,
        TransitEncryptionEnabled=True,
        AtRestEncryptionEnabled=True,
        Tags=_tags(tags),
    )
    if num_shards > 1:
        kwargs['AutomaticFailoverEnabled'] = True
    if auth_token:
        token = _secrets_mod.token_urlsafe(32)
        kwargs['AuthToken'] = token
        create_secret(auth, f'elasticache/{name}/auth', token)
    try:
        return client.create_replication_group(**kwargs)['ReplicationGroup']
    except client.exceptions.ReplicationGroupAlreadyExistsFault:
        return client.describe_replication_groups(
            ReplicationGroupId=name)['ReplicationGroups'][0]

def redis_conn(auth, name) -> str:
    'Return the rediss:// connection string for the primary endpoint.'
    rg = _elasticache(auth).describe_replication_groups(
        ReplicationGroupId=name)['ReplicationGroups'][0]
    ep = rg['NodeGroups'][0]['PrimaryEndpoint']
    return f'rediss://{ep["Address"]}:{ep["Port"]}'
